## # Outpatient metrics for Model Health System | Extract Transform and Load (ETL)
Notes:
- Refer to Cells 3 and 22- keep date range under review. Currently from date is set at April 2024
- When April 2026 activity is available - changes will be need to reflect the new ICB structures. Currently waiting to confirm steps needed with MHS needed (for example, we may continue to load data for 42 ICBs). TBC

In [0]:
#1 Importing Tools 
import pandas as pd
from datetime import datetime
from openpyxl.styles import NamedStyle

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    when, col, lit, create_map, coalesce, last_day, to_date, concat_ws,
    current_date, add_months, date_format, explode, array, struct
)
from pyspark.sql.types import DateType, StringType, DoubleType
from pyspark import StorageLevel

In [0]:
#Set date ranges
start_month_yyyymm = F.lit("202404")  #change as needed
latest_month_yyyymm = F.lit("202602")  #change as needed

#As above, we are processing the data from April 2024. 
#Data prior to March 2023 is frozen anyway, as data prior to this is based on the old monthly SUS extracts, whereas data after March 2023 is based on dailySUS submissions (monthly snapshot). Historical values are less likely to change and/or less likely to be of interest to MHS users. 
#This was last updated on 14th April 2026. Keep under Review. 


In [0]:
#Reduce risk of a timeout by increasing limit to 30 minutes
spark.conf.set("spark.databricks.execution.timeout", "1800")

In [0]:
%skip
#3 Loading the master hierarchies table from the lake mart (via ADLS path)

# Set connection settings for prov ref data
analysisLake = "udalstdatacuratedprod.dfs.core.windows.net"
analysisContainer = "reporting"
provFile = "/unrestricted/reference/UKHD/ODS/Provider_Hierarchies_ICB"

provPath = f"abfss://{analysisContainer}@{analysisLake}{provFile}"

# Load provider reference data
df_master_hierarchies = (
    spark.read
         .option("header", "true")
         .option("recursiveFileLookup", "true")
         .parquet(provPath)
)

# Display sample
display(df_master_hierarchies.limit(10))

# Count active records (where Effective_To is NULL)
active_count = df_master_hierarchies.filter(F.col("Effective_To").isNull()).count()

print(f"Number of rows in master hierarchies (active records): {active_count}")

**Extract and transform the Outpatient Activity data**

In [0]:

#Reference data last reviewed 14th April 2026.

# PROVIDER HIERARCHIES: Bring in the local version of ProviderHierarchies (this version include 42 ICBs, as per ODS structure in 2025/26)
# File path to Lake. Load provider reference data (CSV instead of parquet)
provPath = "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/2526_ProviderHierarchiesExtract.csv"
df_master_hierarchies = (spark.read.option("header", "true").option("inferSchema", "true").csv(provPath))

# ICB TO REGION CODES: This ensures that all ICBs are aligned consistency to a region code. In the processing steps, provider codes > ICB codes. ICB Codes > Region codes. 
df_icb_region = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_ICB_Region_DisplayNames.csv")  

# List of MERGED PROVIDER CODES
df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/MHS_Merged_Providers.csv")

# OPRT METRIC List, including internal MHS ID codes required for ingestion 
mhs_metric_list = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/List of metrics for workflow V1.3.csv")

# ALLOWABLE ORG CODES - MHS does not ingest data for all provider codes. A locally held list has been created by the OPRT Analytical team. It lists Regions, ICB and 'main' Providers of Outpatient services. This includes acute and non-acute providers. 
mhs_allowable_orgs = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/Allowable_Org_Codes_Status.csv")

#BROWSE REFERENCE DATA. 
# This step CREATES 5 Datframes to be used in the later processing steps.

#display (df_master_hierarchies.limit(10))
#display(df_icb_region.limit(10))
#display(df_merged_providers.limit(10))
#display(mhs_metric_list.limit(10))
#display(mhs_allowable_orgs.limit(10))

print(f"Row Count Provider Hierarhices: {df_master_hierarchies.count()}") #expect 133,168 
print(f"Row Count icb_region: {df_icb_region.count()}") #expect 42 
print(f"Row count merged providers: {df_merged_providers.count()}") #expect 39
print(f"Number of rows in mhs_allowable_orgs: {mhs_allowable_orgs.count()}") #expect 241
print(f"Number of rows in mhs_metric_list: {mhs_metric_list.count()}") # expect 1,441


In [0]:
#Loading the Outpatient Activity Data (SUS data)
df_op_activity_snapshot = spark.read.option("header", "true").option("recursiveFileLookup", "true").parquet(
    "abfss://reporting@udalstdatacuratedprod.dfs.core.windows.net/restricted/patientlevel/MESH/OPA/OPA_Core_Monthly_Snapshot/Published/1")

#BROWSE OP DATA
display(df_op_activity_snapshot.limit(10))

In [0]:
#TREATMENT FUNCTION CODES
# Define valid treatment function codes.
# For OP metrics, we create metrics relating to the main TFCS i.e. specialties which are high volume or high profile
VALID_TREATMENT_CODES = [
    '100', '101', '102', '104', '105', '106', '108', '110', '111', '115', '120', '130', '140',
    '144', '145', '301', '302', '303', '307', '320', '330', '340', '361', '400', '410', '420',
    '430', '501', '502', '560', '650']

# This next step adds an extra column into to the dataframe.
# The extra column is called 'Treatment Function Code New' 
# This extra column includes a TFC where it is valid, otherwise it has a text value 'Other' 
# 'Other' activity needs to be included at this step to ensure that 'ALL' activity volumes can be created in STEPxxx
opa_with_tfc = df_op_activity_snapshot.withColumn(
    "Treatment_Function_Code_New",
    when(col("Treatment_Function_Code").isin(VALID_TREATMENT_CODES),
         col("Treatment_Function_Code")
    ).otherwise("Other"))

# For MHS reporting, some individual TFCs need to be grouped and presented as one specialty.
# These groupings are based on a) RTT mappings OR b) advice from GIRFT colleagues  
# Add Treatment_Function_Group (GS / OMFS / T&O / code)
opa_with_groups = opa_with_tfc.withColumn(
    "Treatment_Function_Group",
    when(col("Treatment_Function_Code_New").isin("100", "102", "104", "105", "106"), "GS") #General Surgery
    .when(col("Treatment_Function_Code_New").isin("140", "144", "145"), "OMFS") #Oral Max Fax
    .when(col("Treatment_Function_Code_New").isin("110", "111", "115"), "T&O") #Trauma & Orthopaedics
    .otherwise(col("Treatment_Function_Code_New")))  #All other TFCs (including 'Other'))

#BROWSE OPA DATA with additional columns relating to the TFC processing
display(opa_with_groups.limit(10))

#** TFC Group is the field required for the later steps**

In [0]:
#APPLY DATE RANGE FILTERS TO THE OPA DATA AND CRITERIA TO IDENTIFY OP ACTIVITY FOR METRICS  

#FILTERS FOR OPA - nhs fUNDED, NO RTT EVENTS, NO 812
# Filter dataset (years, admin category, TFC, attendance)
opa_filtered = opa_with_groups.filter(        
    (col("Administrative_Category") == "01") & #INDICATES NHS FUNDED CARE
    (col("Treatment_Function_Code") != "812") & #REMOVED ACTIVITY RELATING TO TFC 812 - too much variation in use 
    (col("First_Attendance").isin("1", "2", "3", "4"))) #Includes all attendances which are not RTT Administrative events

# START AND END DATES SET IN Cell 2
opa_filtered = opa_filtered.filter((col("Der_Activity_Month") >= start_month_yyyymm) &
    (col("Der_Activity_Month") <= latest_month_yyyymm))

In [0]:
#CREATE the VARIABLES required for metric calculations
# Aggregate metrics by month, provider, and TFC group
opa_agg = opa_filtered.groupBy(
    "Der_Activity_Month",
    "Der_Provider_Code",
    "Treatment_Function_Group"
).agg(
    # FIRST/FU 
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("1", "2", "3", "4")), 1).otherwise(0)).alias("All_Total"),
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("1", "3")), 1).otherwise(0)).alias("All_First"),
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("2", "4")), 1).otherwise(0)).alias("All_FU"),

    #With or Without PROCEDURES
    F.sum(when((col("Der_Number_Procedure") > 0) & (col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("1", "2", "3", "4")), 1).otherwise(0)).alias("All_Proc"),
    F.sum(when((col("Der_Number_Procedure") == 0) & (col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("1", "2", "3", "4")), 1).otherwise(0)).alias("All_NoProc"),
    F.sum(when((col("Der_Number_Procedure") > 0) & (col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("2", "4")), 1).otherwise(0)).alias("All_FU_Proc"),
    F.sum(when((col("Der_Number_Procedure") == 0) & (col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("2", "4")), 1).otherwise(0)).alias("All_FU_NoProc"),

    # Face-to-face
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("1", "2")), 1).otherwise(0)).alias("F2F_Total"),
    F.sum(when( (col("Attendance_Status").isin("5", "6")) & (col("First_Attendance") == "1"), 1).otherwise(0)).alias("F2F_First"),
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance") == "2"), 1).otherwise(0)).alias("F2F_FU"),
    
    # Remote
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance").isin("3", "4")), 1).otherwise(0)).alias("Remote_Total"),
    F.sum(when((col("Attendance_Status").isin("5", "6")) & (col("First_Attendance") == "3"), 1).otherwise(0)).alias("Remote_First"),
    F.sum(when((col("Attendance_Status").isin("5", "6")) &(col("First_Attendance") == "4"), 1).otherwise(0)).alias("Remote_FU"),

    # Did not attends (DNAs)
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("1", "2", "3", "4")), 1).otherwise(0)).alias("All_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("1", "3")), 1).otherwise(0)).alias("All_First_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("2", "4")), 1).otherwise(0)).alias("All_FU_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("1", "2")), 1).otherwise(0)).alias("F2F_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("3", "4")), 1).otherwise(0)).alias("Remote_DNA"),

    # 2WW DNA
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("1", "2", "3", "4")) &(col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_2WW_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("1", "3")) & (col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_First_2WW_DNA"),
    F.sum(when((col("Attendance_Status").isin("3", "7")) & (col("First_Attendance").isin("2", "4")) & (col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_FU_2WW_DNA"),

    # All 2WW 
    F.sum(when((col("Attendance_Status").isin("5", "6", "3", "7")) & (col("First_Attendance").isin("1", "2", "3", "4")) & (col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_2WW"),
    F.sum(when((col("Attendance_Status").isin("5", "6", "3", "7")) & (col("First_Attendance").isin("1", "3")) & (col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_First_2WW"),
    F.sum(when((col("Attendance_Status").isin("5", "6", "3", "7")) & (col("First_Attendance").isin("2", "4")) & (col("Priority_Type") == "3"), 1).otherwise(0)).alias("All_FU_2WW"))
          
display(opa_agg.limit(10))


In [0]:
#CREATE A ROW FOR 'ALL SPECIALTIES' within each metric/org and month 
#For example, this means that report on total OP activity for a provider 

METRIC_COLS = [c for c in opa_agg.columns if c not in ["Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group"]]

opa_all_tfc = opa_agg.groupBy("Der_Activity_Month", "Der_Provider_Code").agg(
    *[F.sum(col(c)).alias(c) for c in METRIC_COLS]).withColumn("Treatment_Function_Group", lit("All"))

opa_final = opa_agg.unionByName(opa_all_tfc)
# Order by results
opa_final_ordered = opa_final.orderBy("Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group")

#BROWSE DATA 
display(opa_final_ordered.limit(10))


In [0]:

# POVIDER BASE CODE - REMOVED TRAILING 00's
# Tidy up Provider codes (eg. to make 5 character ODS codes into 3 characters for example.
# This ensures that activity reported against a site code is aligned to the correct Provider code eg R0A01 should be aligned to R0A 
#Creates a column called 'Provider Base Code'. This removes trailing zeroes from the 'Der_Provider_Code' field from the SUS Dataset. 
#With the exception of NQT00 which is a valid ODS code. 

opa_final_ordered = opa_final_ordered.withColumn(
    "Provider_Base_Code",
    when((col("Der_Provider_Code").substr(4, 2) == "00") & (col("Der_Provider_Code") != "NQT00"),   # e.g. ABC00. NQT00 is an exception to this rule. 
        col("Der_Provider_Code").substr(1, 3)                                                       # -> ABC
    ).otherwise(col("Der_Provider_Code")))                                                          # everything else is unchanged


# APPLIES MERGERS 
# This section creates a dictionary mapping old provider codes to new provider codes, then constructs a Spark SQL map expression (mapping_expr) for use in withColumn. This allows you to replace any old provider code with its new code in a DataFrame column.
#Creating the dictionary 
provider_code_mapping_dict = {row["Old_Provider_Code"]: row["New_Provider_Code"]
    for row in df_merged_providers.select("Old_Provider_Code", "New_Provider_Code").distinct().collect()}

mapping_list = []
for k, v in provider_code_mapping_dict.items():
    mapping_list.append(lit(k))
    mapping_list.append(lit(v))
mapping_expr = create_map(*mapping_list)

# Add a new field call"Adj_Org_Code" based on Provider_Base_Code & with mergers applied 
opa_final_ordered_with_adj = opa_final_ordered.withColumn("Adj_Org_Code",
    coalesce(mapping_expr.getItem(col("Provider_Base_Code")), col("Provider_Base_Code")))

#kc i think this can be removed and adj code used???
# Create a column called 'Org_Code_for_Join'. This includes all of the adjustments above. 
# opa_with_join_code = opa_final_ordered_with_adj.withColumn(
#     "Org_Code_For_Join", when(col("Adj_Org_Code") == "NQT00", col("Adj_Org_Code"))
#     .when((F.length(col("Adj_Org_Code")) == 5) & col("Adj_Org_Code").endswith("00"),
#         col("Adj_Org_Code").substr(1, 3)).otherwise(col("Adj_Org_Code")))

# Summary of this step: 
# Der Provider Code > Provider Base Code > Adj Org Code 
# From SUS > Removes trailing zeros & Applies Mergers > Create final field for use in next step. 


In [0]:

#Join to master hierarchies to get ICB (ICB_Code)
opa_with_icb = opa_final_ordered_with_adj.join(df_master_hierarchies.select(
        F.col("Organisation_Code").alias("join_org_code"),
        F.col("ICB_Code").alias("ICB")),
    opa_final_ordered_with_adj["Adj_Org_Code"] == F.col("join_org_code"),"left").drop("join_org_code")



# 12. Join to df_icb_region to get Region_Code
opa_with_icb_region = opa_with_icb.join(
    df_icb_region.select(
        F.col("ICB_Code").alias("join_icb"),
        F.col("Region_Code")
    ),
    opa_with_icb["ICB"] == F.col("join_icb"),
    "left"
).drop("join_icb")

display(spark.createDataFrame([(opa_with_icb_region.count(),)], ["opa_with_icb_region_count"]))
display(opa_with_icb_region.limit(10))

In [0]:
# Create a new column called Der_Activity_Month_Date format 
# Der Activity month is extracted from SUS in the from of YYYYMM.
# This step changes the date to be the month end date

opa_with_icb_region = opa_with_icb_region.withColumn(
    "Der_Activity_Month_Date", 
       last_day(to_date(concat_ws("-", col("Der_Activity_Month").substr(1, 4), col("Der_Activity_Month").substr(5, 2), lit("01")))))

#Browse results as QA step 
display(opa_with_icb_region.select("Der_Activity_Month", "Der_Activity_Month_Date").limit(10))

In [0]:
# Aggregate to final level & sort
# This step aggregates the outpatient activity data to a final reporting level.
# It groups the data by reporting month, specialty group, region, ICB, and adjusted org code,
# then sums all metric columns for each group. The result is sorted for easier downstream use.

id_cols = ["Der_Activity_Month_Date", "Treatment_Function_Group", "Region_Code", "ICB", "Adj_Org_Code"]

metric_cols = [
    c for c in opa_with_icb_region.columns
    if c not in id_cols + [
        "Der_Activity_Month",
        "Der_Provider_Code",
        "Treatment_Function_Code_New",
        "Provider_Base_Code"]]

opa_final_processed = (
    opa_with_icb_region
    .groupBy(*[F.col(c) for c in id_cols])
    .agg(*[F.sum(F.col(c)).alias(c) for c in metric_cols])
    .orderBy("Der_Activity_Month_Date", "Region_Code", "ICB", "Adj_Org_Code", "Treatment_Function_Group")
)

display(opa_final_processed.limit(10))
print(f"opa_final_processed: {opa_final_processed.count()}")


In [0]:
#10 —Create % metrics - this step add the extra metrics to the wide table
df = df = opa_final_processed 

def metric_calcs(df, new_col, expr_fn, required_cols):
    if all(c in df.columns for c in required_cols):
        return df.withColumn(new_col, expr_fn(df))
    else:
        return df.withColumn(new_col, F.lit(None))

# Define new percentage/rate metrics to add, with their calculation logic and required columns
# The expr_fn is a lambda function that takes the DataFrame (d) as input and returns a column expression.

metrics = [
    # DNA Metrics
   ("All_DNA_Over_All_Total", lambda d: F.when((F.col("All_Total") + F.col("All_DNA")) != 0,(F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
   ), ["All_Total", "All_DNA"]),

   ("All_DNA_Over_All_Total_IG", lambda d: F.when((F.col("All_Total") + F.col("All_DNA")) != 0,(F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),

    ("All_First_DNA_Over_All_First", lambda d: F.when((F.col("All_First") + F.col("All_First_DNA")) != 0,(F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100), ["All_First", "All_First_DNA"]),

    ("All_First_DNA_Over_All_First_IG", lambda d: F.when((F.col("All_First") + F.col("All_First_DNA")) != 0,(F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100), ["All_First", "All_First_DNA"]),

    ("All_FU_DNA_Over_All_FU", lambda d: F.when((F.col("All_FU") + F.col("All_FU_DNA")) != 0,(F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),

    ("All_FU_DNA_Over_All_FU_IG", lambda d: F.when((F.col("All_FU") + F.col("All_FU_DNA")) != 0,(F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100), ["All_FU", "All_FU_DNA"]),

    # 2WW Metrics
    ("All_2WW_DNA_Over_All_2WW", lambda d: F.when((F.col("All_2WW") != 0) & (F.col("All_2WW").isNotNull()),
        (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100), ["All_2WW_DNA", "All_2WW"]),
    
    ("All_FU_2WW_DNA_Over_All_FU_2WW", lambda d: F.when((F.col("All_FU_2WW") != 0) & (F.col("All_FU_2WW").isNotNull()),(F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100), ["All_FU_2WW_DNA", "All_FU_2WW"]),

    ("All_First_2WW_DNA_Over_All_First_2WW", lambda d: F.when((F.col("All_First_2WW") != 0) & (F.col("All_First_2WW").isNotNull()),(F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100), ["All_First_2WW_DNA", "All_First_2WW"]),

    # FU Metrics
    ("All_FU_NoProc_Over_All_FU", lambda d: F.when(F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100), ["All_FU_NoProc", "All_FU"]),

    ("All_FU_Proc_Over_All_FU", lambda d: F.when(F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100), ["All_FU_Proc", "All_FU"]),

    ("All_FU_To_All_First", lambda d: F.when(F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))), ["All_FU", "All_First"]),

    ("All_FU_Over_All_Total", lambda d: F.when(F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100), ["All_FU", "All_Total"]),

    ("All_First_Over_All_Total", lambda d: F.when(F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100), ["All_First", "All_Total"]),

    ("All_NoProc_Over_All_Total", lambda d: F.when(F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100), ["All_NoProc", "All_Total"]),

    ("All_Proc_Over_All_Total", lambda d: F.when(F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100), ["All_Proc", "All_Total"]),

    # REMOTE METRICS
    ("Remote_Total_Over_All_Total", lambda d: F.when(F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100), ["Remote_Total", "All_Total"]),

    ("Remote_FU_Over_All_FU", lambda d: F.when(F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100), ["Remote_FU", "All_FU"]),
    ("Remote_First_Over_All_First", lambda d: F.when(F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100), ["Remote_First", "All_First"]),
    
    ("F2F_DNA_Over_F2F_Total", lambda d: F.when((F.col("F2F_Total") + F.col("F2F_DNA")) != 0,(F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
    ), ["F2F_Total", "F2F_DNA"]),

    ("Remote_DNA_Over_Remote_Total", lambda d: F.when((F.col("Remote_Total") + F.col("Remote_DNA")) != 0,(F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100), ["Remote_Total", "Remote_DNA"]),
]

# Apply each metric calculation to the DataFrame
for name, expr, req in metrics:
    df = metric_calcs(df, name, expr, req)

# Add simple copy columns (e.g. All_DNA1 = All_DNA)
simple_copies = [
    ("All_DNA1", "All_DNA"),
    ("All_DNA2", "All_DNA"),
    ("All_First1", "All_First"),
    ("All_First2", "All_First"),
    ("All_First3", "All_First"),
    ("All_FU1", "All_FU"),
    ("All_FU2", "All_FU"),
    ("All_FU3", "All_FU"),
    ("All_FU4", "All_FU"),
    ("All_FU5", "All_FU"),
    ("All_Total1", "All_Total"),
    ("All_Total2", "All_Total"),
    ("All_Total3", "All_Total"),
    ("All_Total4", "All_Total"),
    ("All_Total5", "All_Total"),
    ("All_Total6", "All_Total"),
    ("Remote_Total1", "Remote_Total"),
    ("Remote_Total2", "Remote_Total"),
]
for newc, base in simple_copies:
    if base in df.columns:
        df = df.withColumn(newc, F.col(base))
    else:
        df = df.withColumn(newc, F.lit(None))

# Add combo columns (e.g. All_First_plus_All_First_DNA = All_First + All_First_DNA)
combos = [
    ("All_First_plus_All_First_DNA", ["All_First", "All_First_DNA"]),
    ("All_FU_plus_All_FU_DNA", ["All_FU", "All_FU_DNA"]),
    ("All_Total_plus_All_DNA", ["All_Total", "All_DNA"]),
    ("F2F_Total_plus_F2F_DNA", ["F2F_Total", "F2F_DNA"]),
    ("Remote_Total_plus_Remote_DNA", ["Remote_Total", "Remote_DNA"]),
]
for newc, cols in combos:
    if all(c in df.columns for c in cols):
        df = df.withColumn(newc, F.col(cols[0]).cast("long") + F.col(cols[1]).cast("long"))
    else:
        df = df.withColumn(newc, F.lit(None))

# Drop unnecessary columns
cols_to_drop = [c for c in ["Der_Activity_Month"] if c in df.columns]
opa_final_with_added_metrics = df.drop(*cols_to_drop)

display(opa_final_with_added_metrics.limit(10))
print(f"opa_final_with_added_metrics: {opa_final_with_added_metrics.count()} rows")

In [0]:
# This cell aggregates outpatient metrics to three levels: Org (provider), ICB, and Region.
# It calculates sums for all base count columns at each level, then adds derived rate/percentage metrics.
# It also creates duplicate and combo columns for downstream use, and standardises the code column for each level.
# The result is a single DataFrame (final_output) with all levels combined, ready for reshaping or export.

df_org = opa_final_with_added_metrics

# Sets the metric columns to be aggregated to ICB/Region
BASE_COUNT_COLS = [
    "All_Total","All_First","All_FU","All_Proc","All_NoProc","All_FU_Proc","All_FU_NoProc","F2F_Total","F2F_First","F2F_FU","F2F_DNA","Remote_Total","Remote_First","Remote_FU","Remote_DNA","All_DNA","All_First_DNA","All_FU_DNA","All_2WW","All_First_2WW","All_FU_2WW","All_2WW_DNA","All_First_2WW_DNA","All_FU_2WW_DNA"]
# Step to ensure that columns that actually exist are aggregated
count_cols = [c for c in BASE_COUNT_COLS if c in df_org.columns]

# Define the logic needed for the metrics
#These columumns were created in the dataframe in Step 11. 

def add_rate_metrics(df):
    return (df
        # DNA metrics
        .withColumn("All_DNA_Over_All_Total", F.when((F.col("All_Total") + F.col("All_DNA")) != 0,(F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100))

        .withColumn("All_DNA_Over_All_Total_IG", F.col("All_DNA_Over_All_Total")).withColumn("All_First_DNA_Over_All_First", F.when((F.col("All_First") + F.col("All_First_DNA")) != 0,(F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100))

        .withColumn("All_First_DNA_Over_All_First_IG", F.col("All_First_DNA_Over_All_First")).withColumn("All_FU_DNA_Over_All_FU", F.when((F.col("All_FU") + F.col("All_FU_DNA")) != 0,(F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100))

        .withColumn("All_FU_DNA_Over_All_FU_IG", F.col("All_FU_DNA_Over_All_FU"))
        
        # FU metrics
        .withColumn("All_FU_NoProc_Over_All_FU", F.when(F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100))
        .withColumn("All_FU_Proc_Over_All_FU", F.when(F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100))
        .withColumn("All_FU_To_All_First", F.when(F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))))
        
        # 2WW
        .withColumn("All_2WW_DNA_Over_All_2WW", F.when(F.col("All_2WW") != 0, (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100))
        .withColumn("All_First_2WW_DNA_Over_All_First_2WW", F.when(F.col("All_First_2WW") != 0, (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100))
        .withColumn("All_FU_2WW_DNA_Over_All_FU_2WW", F.when(F.col("All_FU_2WW") != 0, (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100))
       
        # Mix shares
        .withColumn("All_FU_Over_All_Total", F.when(F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100))
        .withColumn("All_First_Over_All_Total", F.when(F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100))
        .withColumn("All_NoProc_Over_All_Total", F.when(F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100))
        .withColumn("All_Proc_Over_All_Total", F.when(F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100))
        
        #Remote metric
        .withColumn("Remote_Total_Over_All_Total", F.when(F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100))
        .withColumn("Remote_FU_Over_All_FU", F.when(F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100))
        .withColumn("Remote_First_Over_All_First", F.when(F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100))
        .withColumn("Remote_DNA_Over_Remote_Total", F.when((F.col("Remote_Total") + F.col("Remote_DNA")) != 0,(F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100))
        .withColumn("F2F_DNA_Over_F2F_Total", F.when((F.col("F2F_Total") + F.col("F2F_DNA")) != 0, (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100)))

# Aggregate Variables: 
df_icb = (df_org.groupBy("Der_Activity_Month_Date", "ICB", "Treatment_Function_Group").agg(*[F.sum(col(c)).alias(c) for c in count_cols]))
df_region = (df_org.groupBy("Der_Activity_Month_Date", "Region_Code", "Treatment_Function_Group").agg(*[F.sum(col(c)).alias(c) for c in count_cols]))

# Add Metric calculations and an additional field to denote 'Org Level' 
df_icb = add_rate_metrics(df_icb).withColumn("Level", lit("ICB"))
df_region = add_rate_metrics(df_region).withColumn("Level", lit("Region"))

# For metrics calculated at org level, add a value to denote 'Org Level' 
df_org = df_org.withColumn("Level", lit("Org"))

# Combine all levels
final_output = (df_org.unionByName(df_icb, allowMissingColumns=True).unionByName(df_region, allowMissingColumns=True)
)

# Add a column called 'Adj Org COde Final - this selects the relevant column to use (i.e Adj org code, ICB or Region Code)
final_output = final_output.withColumn("Adj_Org_Code_Final",
    when(col("Level") == "Org", col("Adj_Org_Code"))
    .when(col("Level") == "ICB", col("ICB"))
    .when(col("Level") == "Region", col("Region_Code"))
)

# Creates duplciates of columns in dataset where metrics are to be repeated
copy_map = {
    "All_DNA1": "All_DNA",
    "All_DNA2": "All_DNA",
    "All_FU1": "All_FU",
    "All_FU2": "All_FU",
    "All_FU3": "All_FU",
    "All_FU4": "All_FU",
    "All_FU5": "All_FU",
    "All_First1": "All_First",
    "All_First2": "All_First",
    "All_First3": "All_First",
    "All_Total1": "All_Total",
    "All_Total2": "All_Total",
    "All_Total3": "All_Total",
    "All_Total4": "All_Total",
    "All_Total5": "All_Total",
    "All_Total6": "All_Total",
    "Remote_Total1": "Remote_Total",
    "Remote_Total2": "Remote_Total"
}

for newc, base in copy_map.items():
    if base in final_output.columns:
        final_output = final_output.withColumn(newc, col(base))

combo_pairs = {
    "All_First_plus_All_First_DNA": ("All_First", "All_First_DNA"),
    "All_FU_plus_All_FU_DNA": ("All_FU", "All_FU_DNA"),
    "All_Total_plus_All_DNA": ("All_Total", "All_DNA"),
    "F2F_Total_plus_F2F_DNA": ("F2F_Total", "F2F_DNA"),
    "Remote_Total_plus_Remote_DNA": ("Remote_Total", "Remote_DNA")
}

for newc, (a, b) in combo_pairs.items():
    if a in final_output.columns and b in final_output.columns:
        final_output = final_output.withColumn(newc, col(a).cast("long") + col(b).cast("long"))

# Final preview
display(final_output.limit(20))
print("Container 12 complete —", final_output.count(), "rows")

In [0]:
# Steps in this cell:
# 1. Drops unnecessary columns from the wide table.
# 2. Defines identifier columns to retain (including Level).
# 3. Identifies metric columns to unpivot.
# 4. Defines a helper to safely cast values to double, setting non-numeric to NULL.
# 5. Unpivots (melts) the wide table to long format, with one row per metric per org/date/group/level.
# 6. Filters out rows where the metric value is null.
# 7. Displays a preview and prints row/column counts.
# 8. Sets the output for downstream use.

# 13 – long/skinny OPRT format (Level preserved)
from pyspark.sql.functions import col, lit, explode, array, struct, when
import pyspark.sql.functions as F

# Use the final_output table from container 12 (has Level + Adj_Org_Code_Final)
df_wide = final_output

# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Adj_Org_Code",
    "All_DNA", "All_First", "All_FU",
    "All_Total", "F2F_Total", "Remote_Total"
]
df_wide = df_wide.drop(*[c for c in cols_to_drop if c in df_wide.columns])

# Step 2: Define identifier columns (KEEP Level)
id_cols = [
    "Der_Activity_Month_Date",
    "Region_Code",
    "ICB",
    "Adj_Org_Code_Final",
    "Treatment_Function_Group",
    "Level",
]

# Step 3: Identify metric columns (exclude identifiers)
metric_cols = [c for c in df_wide.columns if c not in id_cols]

# Helper: safe numeric cast under ANSI mode, If value looks like a number → cast to double, Else → NULL

def safe_numeric(colname: str):
    return (
        when(
            col(colname).rlike(r'^[-+]?[0-9]*\.?[0-9]+$'),
            col(colname).cast("double")
        )
        .otherwise(F.lit(None).cast("double"))
        .alias("Metric_Value")
    )

# Step 4: Melt into long/skinny format
opa_oprt_long = (
    df_wide.select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("OPRT_Metric_Name"),
                safe_numeric(c)
            )
            for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.OPRT_Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Drop null metric rows (zeros are kept)
opa_oprt_long = opa_oprt_long.filter(col("Metric_Value").isNotNull())

display(opa_oprt_long.limit(10))
print(f"Container 13 complete — {opa_oprt_long.count()} rows, {len(opa_oprt_long.columns)} columns")

# Ensures next step will refer to cleaned long-format table
final_output = opa_oprt_long

# Display results for Provider code = RCF, METRIC ID = OZ0411, BY REPORTING DATE
display(
    final_output
    .filter(
        (col("Adj_Org_Code_Final") == "RCF") &
        (col("OPRT_Metric_Name") == "Remote_Total1") &
        (col("Treatment_Function_Group") == "All")
    )
    .orderBy("Der_Activity_Month_Date")
)

In [0]:
# 14 Prepare Preview Outpatient Metrics and Map Internal IDs
# This cell prepares the outpatient metrics for preview and mapping to internal metric IDs.
# The output from this step renames the value field as 'PREVIEW' - indicating that this is draft value to be loaded into PREVIEW 


# 1. Filters out 'Other' specialties from the long-format metrics.
df_outpat_metrics = final_output.filter(col("Treatment_Function_Group") != "Other")

# 2. Constructs a combined metric name for joining to the metric ID reference.
df_outpat_metrics = df_outpat_metrics.withColumn(
    "OPRT_Metric_Name_TFC",
    concat_ws("_", col("OPRT_Metric_Name"), col("Treatment_Function_Group")))

# 3. Joins to the metric reference list to map to InternalID.

def normalise_metric_key(col_expr):
    return F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(
                F.lower(F.regexp_replace(F.trim(col_expr), r"\s+", "_")),
                r"[^a-z0-9_]", ""
            ),
            r"_+", "_"
        ),
        r"^_+|_+$", ""
    )

df_outpat_metrics = df_outpat_metrics.withColumn(
    "join_metric",
    normalise_metric_key(col("OPRT_Metric_Name_TFC"))
)


metric_list_cols = [c for c in mhs_metric_list.columns if "OPRT" in c and "TFC" in c]
if len(metric_list_cols) == 0:
    raise ValueError(
        "Could not find OPRT TFC metric column name in mhs_metric_list. Review mhs_metric_list columns."
    )
metric_list_col = metric_list_cols[0]

# Prepare cleaned metric lookup
mhs_metric_list_clean = mhs_metric_list.withColumn(
    "join_metric",
    normalise_metric_key(col(metric_list_col))
)

select_cols = ["join_metric", "InternalID"]
if "Description" in mhs_metric_list_clean.columns:
    select_cols.append("Description")
else:
    print("WARNING: mhs_metric_list does not contain a column named 'Description'.")
mhs_metric_list_clean_sel = mhs_metric_list_clean.select(*select_cols).distinct()

# Optional debug join to inspect unmatched metrics
df_outpat_metrics_left = df_outpat_metrics.join(
    mhs_metric_list_clean_sel,
    on="join_metric",
    how="left")
unmatched = (
    df_outpat_metrics_left
    .filter(col("InternalID").isNull())
    .select("OPRT_Metric_Name_TFC")
    .distinct())

df_outpat_metrics = (
    df_outpat_metrics
    .join(mhs_metric_list_clean_sel, on="join_metric", how="inner")
    .drop("join_metric"))

# 4. Optionally filters to allowable org codes.
if "Org_Code" in mhs_allowable_orgs.columns:
    df_outpat_metrics = df_outpat_metrics.join(
        mhs_allowable_orgs.select(col("Org_Code").alias("allowable_code")),
        df_outpat_metrics["Adj_Org_Code_Final"] == col("allowable_code"),
        "inner"
    ).drop("allowable_code")

# 5. Enforces clean data types and renames columns for downstream use.

df_outpat_metrics = (
    df_outpat_metrics
    .withColumn("Der_Activity_Month_Date", col("Der_Activity_Month_Date").cast(DateType()))
    .withColumn("Adj_Org_Code_Final", col("Adj_Org_Code_Final").cast(StringType()))
    .withColumn("Level", col("Level").cast(StringType()))
    .withColumn("Treatment_Function_Group", col("Treatment_Function_Group").cast(StringType()))
    .withColumn("OPRT_Metric_Name", col("OPRT_Metric_Name").cast(StringType()))
    .withColumn("OPRT_Metric_Name_TFC", col("OPRT_Metric_Name_TFC").cast(StringType()))
    .withColumn("Metric_Value", col("Metric_Value").cast(DoubleType()))
)

df_outpat_metrics = (
    df_outpat_metrics
    .withColumnRenamed("Der_Activity_Month_Date", "reportingDate")
    .withColumnRenamed("Adj_Org_Code_Final", "providerCode")
    .withColumnRenamed("InternalID", "metricID")
    .withColumnRenamed("Metric_Value", "value")
)

# 6. Rounds and selects only required columns.
df_outpat_metrics = df_outpat_metrics.withColumn("value", F.round(col("value"), 2)).select("reportingDate", "providerCode", "metricID", "value")

# Persist once for reuse in later cells
df_outpat_metrics = df_outpat_metrics.persist(StorageLevel.MEMORY_AND_DISK)
_ = df_outpat_metrics.count()

#isplay(df_outpat_metrics.limit(20))

# Display results for Provider code = RCF, METRIC ID = OZ0411, BY REPORTING DATE
display(
    df_outpat_metrics
    .filter(
        (col("providerCode") == "RCF") &
        (col("metricID") == "OZ0411") 
    )
    .orderBy("reportingDate"))

**Ingestion to Model Health System (Net change process)**
Loads values for the OP Metrics from MHS Productions

In [0]:
# Extract Allowable OZ Metric Lists and Pull Matching Production Data
import openpyxl
import sys
sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES

# Create 'metric_lists' from mhs_metric_list, only returning the InternalID field
metric_lists = {
    "all": [row["InternalID"] for row in mhs_metric_list.select("InternalID").distinct().collect() if row["InternalID"] is not None]
}
# Combine all metric IDs from the extracted lists
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of metric IDs:", len(all_metric_ids))
print("Example codes:", all_metric_ids[:10])

if not all_metric_ids:
    raise ValueError("No metric IDs found in the workbook lists.")

# Build SQL IN clause
metric_id_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in all_metric_ids)

# Connect to SQL Server
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

# Query Production data
query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,CAST(mv.[value] AS FLOAT) AS value
      ,m.[InternalID] AS metricID
      ,p.[Code] AS providerCode
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
LEFT JOIN [dbo].[Provider] p
    ON p.ID = mv.ProviderID
WHERE mv.[reportingDate] >= '2024-04-30'
  AND m.[InternalID] IN ({metric_id_sql})
"""

prod_df = mhs_backend_conn.read_from_db(query)

# Rename the value column in prod_df to 'PROD' and round to 2 decimal places
from pyspark.sql.functions import round as spark_round

prod_df = prod_df.withColumn("PROD", spark_round(prod_df["value"], 2)).drop("value")

display(prod_df.limit(10))
prod_row_count = prod_df.count()
display(spark.createDataFrame([(prod_row_count,)], ["row_count"]))


In [0]:
# Join keys: reportingDate, providerCode, metricID
join_keys = ["reportingDate", "providerCode", "metricID"]

# 1) Left only (in prod_df only) - these are the metrics to be removed (deleted)
InProd_DELETE = (
    prod_df.join(df_outpat_metrics, on=join_keys, how="left_anti")
    .select("reportingDate", "providerCode", "metricID")
)
#display(InProd_DELETE)
display(spark.createDataFrame([(InProd_DELETE.count(),)], ["InProd_DELETE_count"]))

# 2) Right only (df_outpat_metrics only)
NewValues_INSERT = df_outpat_metrics.join(prod_df, on=join_keys, how="left_anti")
#display(NewValues_INSERT)
display(spark.createDataFrame([(NewValues_INSERT.count(),)], ["NewValues_INSERT_count"]))

# 3) Inner join (both)
from pyspark.sql.functions import col

NewComparedProd = (
    prod_df.join(df_outpat_metrics, on=join_keys, how="inner")
    .withColumn("difference", col("value") - col("PROD"))
    .filter(col("difference") != 0)
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        "value"
        ))
display(spark.createDataFrame([(NewComparedProd.count(),)], ["NewComparedProd_count"]))
#display(NewComparedProd)

# Append NewValues_INSERT and NewComparedProd into one table called 'INSERT'
INSERT_Metrics = NewValues_INSERT.unionByName(NewComparedProd)
display(spark.createDataFrame([(INSERT_Metrics.count(),)], ["INSERT_Metrics_count"]))


In [0]:
#Delete action for <80k rows
#Value from 20240431 hardcoded into file name - can be changed as needed
import sys
from pyspark.sql import functions as F
sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import INSERT, DELETE

def main():
    max_date = InProd_DELETE.agg(F.max("reportingDate")).collect()[0][0]
    desc = f"20260416_Outpatients_DeleteLegacyValues_{max_date}"

    IngestionHub(InProd_DELETE, desc, True)

    print("Outpatient_metrics sent to IngestionHub")


def IngestionHub(dfih, desc, tf):
    sdf = dfih.withColumn("reportingDate", F.to_date("reportingDate"))

    display(sdf.sample(False, 0.01))
    display(sdf.dtypes)

    MHS_IngestionHub.upload(
        mhs_df=sdf,
        description=desc,
        loaded_by="katie.conners@nhs.net",
        mhs_mode=DELETE,
        skip_existing_data_check=tf
    )

if __name__ == '__main__':
    main()

In [0]:
# INSERT: MHS Ingestion - original script
import sys
from pyspark.sql import functions as F

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')
from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import INSERT, DELETE

def main():
    df_filtered = INSERT_Metrics
    row_count = df_filtered.count()

    if row_count == 0:
        print("ERROR - No rows found' — skipping upload.")
        return

    max_date = (
        df_filtered
        .agg(F.max("reportingDate").alias("max_date"))
        .collect()[0]["max_date"]
    )

    desc = f"20260416_Outpatients_New_Revised_{max_date}"

    InjectionHub(df_filtered, desc, True)

    print(f"OP metrics sent to IngestionHub — {row_count} rows")


def InjectionHub(dfih, desc, tf):

    sdf = dfih.withColumn("reportingDate", F.to_date("reportingDate"))

    display(sdf.sample(False, 0.01))
    display(sdf.dtypes)

    MHS_IngestionHub.upload_many(
        mhs_df=sdf,
        description=desc,
        loaded_by="katie.conners@nhs.net",
        mhs_mode=INSERT,
        skip_existing_data_check=tf
    )


if __name__ == '__main__':
    main()